In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import duckdb
from IPython.display import display, HTML
from sklearn.model_selection import KFold, cross_val_score
from sklearn import linear_model
import statsmodels.api as sm
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import t

In [ ]:
DATASET = os.getenv('WORKSPACE_CDR')

In [ ]:
# Tuple of survey questions that are examined in this analysis
question_concept_ids = (1585729, 1585723, 1585735, 1585754, 1585711, 1585717, 1585760, 1585748, 1585747, 1585741, 40192401, 
                        43530588, 43529977, 1585375, 43528660, 43530402, 40192442, 40192480, 40192388, 40192511, 40192439, 
                        40192528, 40192399, 40192446, 40192507, 40192397, 40192504, 40192398, 40192501, 40192516, 40192390,
                        40192494, 40192452, 40192381, 40192491, 40192419, 40192525, 40192506, 40192449, 40192445, 40192396,
                        40192462)

# Tuple of physical measurements that are examined in this analysis
measurement_concept_ids = (903115, 903126, 903118)

In [ ]:
# Explore unique answers to each question_concept_id

query = f"""
SELECT DISTINCT answer, question_concept_id
FROM {DATASET}.ds_survey
WHERE question_concept_id IN {question_concept_ids}
ORDER BY question_concept_id
"""

df = pd.read_gbq(query, dialect = "standard")
display(HTML(df.to_html()))

In [ ]:
# For each survey question and physical measurement, check to see if each row is a unique person

query = f"""
SELECT COUNT(*) AS numrows, COUNT(DISTINCT person_id) AS people, question_concept_id AS concept_id
FROM {DATASET}.ds_survey
WHERE question_concept_id IN {question_concept_ids}
GROUP BY question_concept_id

UNION ALL

SELECT COUNT(*) AS numrows, COUNT(DISTINCT person_id) AS people, measurement_concept_id AS concept_id
FROM {DATASET}.measurement
WHERE measurement_concept_id IN {measurement_concept_ids}
GROUP BY measurement_concept_id
"""

df = pd.read_gbq(query, dialect = "standard")
display(df)

for i in range(df.shape[0]):
    if df['numrows'][i] != df['people'][i]:
        print(f"For question_concept_id {df['question_concept_id'][i]}, it is not the case that each row is a unique person.")

# Each row is a unique person

In [ ]:
# User-defined functions for the analysis

def answer_cleaning(x):
    if x in ['PMI: Skip', 'No matching concept', 'PMI: Prefer Not To Answer', 'PMI: Dont Know']:
        return np.nan
    elif type(x) == float:
        return x
    elif x[len(x) - 5:len(x)] == 'M ore':
        return '16 or More'
    elif x in ['General Doctor Visits: 1', 'Mental Health Professional Visits: 1']:
        return x
    elif x == 'General Mental Health: Excllent':
        return 'Excellent'
    elif x.find(':') != -1:
        return x[x.find(':') + 2:len(x)]
    else:
        return x
    
# Convert survey answers to numbers that can be used to calculate scores
def answer_transforming(x):
    if x in ['Never pss', 'Very Often pss reverse']:
        return 0
    if x in ['Poor', 'Always', 'Very Severe', 'Not At All', 'Never or almost never', 'less 10k', '10', 'None of the time', 
             'Never loneliness', 'Often loneliness reverse', 'Almost Never pss', 'Fairly Often pss reverse']:
        return 1
    elif x in ['Fair', 'Often', 'Severe', 'A Little', 'Once in a while', 'General Doctor Visits: 1', 
               'Mental Health Professional Visits: 1', '10k 25k', '9', '8', '7', 'A little of the time', 'Rarely loneliness', 
               'Sometimes loneliness reverse', 'Sometimes pss', 'Sometimes pss reverse']:
        return 2
    elif x in ['Good', 'Sometimes', 'Moderate', 'Moderately', 'Some days', '2 to 3', '25k 35k', '6', '5', '4', 
               'Some of the time', 'Sometimes loneliness', 'Rarely loneliness reverse', 'Fairly Often pss', 
               'Almost Never pss reverse']:
        return 3
    elif x in ['Very Good', 'Rarely', 'Mild', 'Mostly', 'Most days', '4 to 5', '35k 50k', '3', '2', '1', 'Most of the time', 
               'Often loneliness', 'Never loneliness reverse', 'Very Often pss', 'Never pss reverse']:
        return 4
    elif x in ['Excellent', 'Never', 'None', 'Completely', 'Every day', '6 to 7', '50k 75k', '0', 'All of the time']:
        return 5
    elif x in ['Many times a day', '8 to 9', '75k 100k']:
        return 6
    elif x in ['10 to 12', '100k 150k']:
        return 7
    elif x in ['13 to 15', '150k 200k']:
        return 8
    elif x in ['16 or More', 'more 200k']:
        return 9
    else:
        return x

# Append the name of the scale to each answer in the Loneliness Scale so that each answer in this scale can be differentiated
def flag_loneliness_qs(x): 
    if pd.isnull(x) == True or x == 'PMI: Skip':
        return x
    else:
        return x + ' loneliness'

# Append the name of the scale to each answer in the PSS Scale so that each answer in this scale can be differentiated
def flag_pss_qs(x):
    if pd.isnull(x) == True or x == 'PMI: Skip':
        return x
    else:
        return x + ' pss'

"""
Append 'reverse' to each answer in the Loneliness or PSS Scales that have a reversed scale so they can be differentiated from 
answers with non-reversed scales
"""
def flag_reverses(x):
    if pd.isnull(x) == True or x == 'PMI: Skip':
        return x
    else:
        return x + ' reverse'
    
# Unfortunately, the formula that maps a raw score value to a t-score value is not public

def t_score_converter_ph(x):
    if x == 4:
        return 16.2
    elif x == 5:
        return 19.9
    elif x == 6:
        return 23.5
    elif x == 7:
        return 26.7
    elif x == 8:
        return 29.6
    elif x == 9:
        return 32.4
    elif x == 10:
        return 34.9
    elif x == 11:
        return 37.4
    elif x == 12:
        return 39.8
    elif x == 13:
        return 42.3
    elif x == 14:
        return 44.9
    elif x == 15:
        return 47.7
    elif x == 16:
        return 50.8
    elif x == 17:
        return 54.1
    elif x == 18:
        return 57.7
    elif x == 19:
        return 61.9
    elif x == 20:
        return 67.7
    
def t_score_converter_mh(x):
    if x == 4:
        return 21.2
    elif x == 5:
        return 25.1
    elif x == 6:
        return 28.4
    elif x == 7:
        return 31.3
    elif x == 8:
        return 33.8
    elif x == 9:
        return 36.3
    elif x == 10:
        return 38.8
    elif x == 11:
        return 41.1
    elif x == 12:
        return 43.5
    elif x == 13:
        return 45.8
    elif x == 14:
        return 48.3
    elif x == 15:
        return 50.8
    elif x == 16:
        return 53.3
    elif x == 17:
        return 56.0
    elif x == 18:
        return 59.0
    elif x == 19:
        return 62.5
    elif x == 20:
        return 67.6
    
def calculate_p_value(x):
    # Used formula from https://support.minitab.com/en-us/minitab/help-and-how-to/statistics/basic-statistics/how-to/correlation/methods-and-formulas/methods-and-formulas/#p-value
    r = pd.to_numeric(x[1:x.find(',')])
    n = pd.to_numeric(x[x.find(',') + 2:len(x) - 1])
    t_stat = (r * np.sqrt(n - 2))/np.sqrt(1 - (r ** 2))
    return 2 * (1 - t.cdf(abs(t_stat), n - 2))

"""
leap year logic added (adding leap year logic did not result in any notable changes in sample size, mean age, or standard 
deviation of age for 30 day sample)
"""
# number of people in 30 day sample changed from 323378 to 313444
# mean age of people in 30 day sample changed from 51.4 to 51.3
# standard deviation of age for 30 day sample changed from 16.63 to 16.62

def date_cleaning(x):

    if pd.isnull(x) == True:
        x = '0000-00-00' # This is not messing up the results because this function is only used to ensure that physical measurements are within 30 days of PROMIS variables
    else:
        x = str(x)

    year = ((pd.to_numeric(x[0:4]) - 1) * 365) + ((pd.to_numeric(x[0:4]) - 1) // 4)
    month = pd.to_numeric(x[5:7])
    day = pd.to_numeric(x[8:10])
    days = year + day

    for i in range(2, month + 1):
        if i in [2, 4, 6, 8, 9, 11]:
            days += 31
        if i in [5, 7, 10, 12]:
            days += 30
        if i == 3 and pd.to_numeric(x[0:4]) % 4 != 0:
            days += 28
        else:
            days += 29
            
    return days

In [ ]:
# Load in the data

query_main = f"""
SELECT survey_datetime, answer, person_id, question_concept_id
FROM {DATASET}.ds_survey
WHERE question_concept_id IN {question_concept_ids}

UNION ALL

SELECT measurement_datetime, CAST(value_as_number AS STRING), person_id, measurement_concept_id
FROM {DATASET}.measurement
WHERE measurement_concept_id IN {measurement_concept_ids}
"""
df_main = pd.read_gbq(query_main, dialect = "standard")

query_demographics = f"""
SELECT DISTINCT P.person_id, P.date_of_birth, P.race, P.ethnicity, P.sex_at_birth
FROM {DATASET}.ds_person P, {DATASET}.ds_survey S
WHERE P.person_id = S.person_id AND S.question_concept_id IN {question_concept_ids}
"""
df_demographics = pd.read_gbq(query_demographics, dialect = "standard")

In [ ]:
# Clean and transform the data

df_date = df_main[['survey_datetime', 'person_id', 'question_concept_id']]
df_main = df_main.drop(columns = 'survey_datetime')
df_main = df_main.pivot(index = 'person_id', columns = 'question_concept_id', values = 'answer')
df_date = df_date.pivot(index = 'person_id', columns = 'question_concept_id', values = 'survey_datetime')
df_date = df_date.rename(columns = {col: f"{col}_datetime" for col in question_concept_ids})
df_date = df_date.drop(columns = [f"{col}_datetime" for col in question_concept_ids[2:5]])
df_date = df_date.drop(columns = [f"{col}_datetime" for col in question_concept_ids[6:16]])
df_date = df_date.drop(columns = [f"{col}_datetime" for col in question_concept_ids[17:len(question_concept_ids)]])
df_date = df_date.rename(columns = {col: f"{col}_datetime" for col in measurement_concept_ids})
df_demographics.index = df_demographics['person_id']
df_demographics = df_demographics.drop(columns = 'person_id')
df = pd.concat([df_main, df_date, df_demographics], axis = 1)

# Flag each of these because they each contain answers that appear in multiple questions but are scored differently
for i in question_concept_ids[24:32]:
    df[i] = df[i].apply(flag_loneliness_qs)
for i in question_concept_ids[32:len(question_concept_ids)]:
    df[i] = df[i].apply(flag_pss_qs)
for i in (40192504, 40192516, 40192525, 40192449, 40192445):
    df[i] = df[i].apply(flag_reverses)

"""
Could not place this before applying the flag loneliness, pss, and reverses function calls because answer_transforming 
requires that these functions already be called
"""
for i in question_concept_ids:
    df[i] = df[i].apply(answer_cleaning)
    df[i] = df[i].apply(answer_transforming)
for i in measurement_concept_ids:
    df[i] = np.float64(df[i])
    
# For doctor and mental health visit questions, add back the zeros
df[43530588] = np.where(df[43528660] == 'No', 1, df[43530588])
df[43529977] = np.where(df[43530402] == 'No', 1, df[43529977])
    
df = df.rename(columns = {1585729:'In general, how would you rate your mental health, including your mood and your ability to think?',
                          1585723:'In general, how would you rate your physical health?',
                          1585735:'In general, how would you rate your satisfaction with your social activities and relationships?',
                          1585754:'In general, please rate how well you carry out your usual social roles.',
                          1585711:'In general, would you say your health is:',
                          1585717:'In general, would you say your quality of life is:',
                          1585760:'In the past 7 days, how often have you been bothered by emotional problems such as feeling anxious, depressed or irritable?',
                          1585748:'In the past 7 days, how would you rate your fatigue on average?',
                          1585741:'To what extent are you able to carry out your everyday physical activities such as walking, climbing stairs, carrying groceries, or moving a chair?',
                          1585747:'In the past 7 days, how would you rate your pain on average?',
                          40192401:'How often do you feel deep inner peace or harmony?',
                          43530588:'What is the total number of general doctor visits you made in the last 12 months?',
                          43529977:'What is the total number of visits to a mental health professional that you made in the last 12 months?',
                          1585375:'What is your annual household income from all sources?',
                          903115:'Computed diastolic blood pressure, mean of 2nd and 3rd measures',
                          903126:'Computed heart rate, mean of 2nd and 3rd measures',
                          903118:'Computed systolic blood pressure, mean of 2nd and 3rd measures'})

In [ ]:
# Raw sums and T-scores
df['Global Physical Health Sum'] = df['In the past 7 days, how would you rate your pain on average?'] + df['To what extent are you able to carry out your everyday physical activities such as walking, climbing stairs, carrying groceries, or moving a chair?'] + df['In general, how would you rate your physical health?'] + df['In the past 7 days, how would you rate your fatigue on average?']
df['Global Mental Health Sum'] = df['In general, would you say your quality of life is:'] + df['In general, how would you rate your mental health, including your mood and your ability to think?'] + df['In general, how would you rate your satisfaction with your social activities and relationships?'] + df['In the past 7 days, how often have you been bothered by emotional problems such as feeling anxious, depressed or irritable?']
df['Global Physical Health T-Score'] = df['Global Physical Health Sum'].apply(t_score_converter_ph)
df['Global Mental Health T-Score'] = df['Global Mental Health Sum'].apply(t_score_converter_mh)

# Social support score
df['Social support score'] = 0
for i in question_concept_ids[16:24]:
    df['Social support score'] += df[i]
df['Social support score'] = 100 * ((df['Social support score'] - 8)/32)

# Loneliness score
df['Loneliness score'] = 0
for i in question_concept_ids[24:33]:
    df['Loneliness score'] += df[i]
df['Loneliness score'] = ((df['Loneliness score']/8) - 1) * (100/3)

# PSS (Perceived Stress Score)
df['PSS'] = 0
for i in question_concept_ids[33:len(question_concept_ids)]:
    df['PSS'] += df[i]
    
# Drop columns that are no longer needed
df = df.drop(columns = list(question_concept_ids[14:len(question_concept_ids)]))

# List of raw sums
sums = ['Global Physical Health Sum',
        'Global Mental Health Sum']

# List of T-scores
tscores = ['Global Physical Health T-Score',
           'Global Mental Health T-Score']

In [ ]:
# Bigger sample
df_big = duckdb.query(f"""
SELECT *
FROM df
WHERE "In the past 7 days, how would you rate your pain on average?" IS NOT NULL OR 
    "To what extent are you able to carry out your everyday physical activities such as walking, climbing stairs, carrying groceries, or moving a chair?" IS NOT NULL OR 
    "In general, how would you rate your physical health?" IS NOT NULL OR 
    "In the past 7 days, how would you rate your fatigue on average?" IS NOT NULL OR 
    "In general, would you say your quality of life is:" IS NOT NULL OR 
    "In general, how would you rate your mental health, including your mood and your ability to think?" IS NOT NULL OR 
    "In general, how would you rate your satisfaction with your social activities and relationships?" IS NOT NULL OR 
    "In the past 7 days, how often have you been bothered by emotional problems such as feeling anxious, depressed or irritable?" IS NOT NULL
""").df()

# Smaller sample
df_small = duckdb.query(f"""
SELECT *
FROM df_big
WHERE "Global Physical Health Sum" IS NOT NULL AND "Global Mental Health Sum" IS NOT NULL
""").df()

# Sample that only includes people who had a physical measurement within 30 days (we chose to use this sample)

# Begin setting up 30 day sample
df_30_days = df_small.copy()

for i in (1585723, 1585717, 40192442):
    df_30_days[f"{i}_datetime"] = df_30_days[f"{i}_datetime"].apply(date_cleaning)
for i in measurement_concept_ids:
    df_30_days[f"{i}_datetime"] = df_30_days[f"{i}_datetime"].apply(date_cleaning)

df_30_days.dropna(subset = ['Social support score'], inplace = True)

print(f"Total number of people in 30 day sample (before filtering for the 30-day window): {df_30_days.shape[0]}")

In [ ]:
"""
For each of the below clauses, produce a histogram showing the number of people in the sample that meet the criteria for 
different length (in number of days) windows
"""
clauses = ('ABS("903115_datetime" - "1585723_datetime") <=', 
           'ABS("903118_datetime" - "1585723_datetime") <=', 
           'ABS("903126_datetime" - "1585723_datetime") <=',
           'ABS("903115_datetime" - "1585717_datetime") <=',
           'ABS("903118_datetime" - "1585717_datetime") <=',
           'ABS("903126_datetime" - "1585717_datetime") <=',
           'ABS("40192442_datetime" - "1585723_datetime") <=',
           'ABS("40192442_datetime" - "1585717_datetime") <=')

for clause in clauses:

    for x in range(0, 100, 10):
        if x == 0:
            histogram_df = duckdb.query(f"""
            SELECT {clause[4:clause.find(' ')]}, {clause[clause.find(' ') + 3:clause.find(')')]}
            FROM df_30_days
            WHERE {clause} {x}
            """).df()
            histogram_df['x'] = x
            histogram_df = histogram_df.drop(columns = ['903115_datetime', 
                                                        '903118_datetime', 
                                                        '903126_datetime', 
                                                        '40192442_datetime',
                                                        '1585723_datetime',
                                                        '1585717_datetime'], 
                                             errors = 'ignore')
        else:
            histogram_df_append = duckdb.query(f"""
            SELECT {clause[4:clause.find(' ')]}, {clause[clause.find(' ') + 3:clause.find(')')]}
            FROM df_30_days
            WHERE {clause} {x}
            """).df()
            histogram_df_append['x'] = x
            histogram_df_append = histogram_df_append.drop(columns = ['903115_datetime', 
                                                                      '903118_datetime', 
                                                                      '903126_datetime', 
                                                                      '40192442_datetime',
                                                                      '1585723_datetime',
                                                                      '1585717_datetime'], 
                                                           errors = 'ignore')
            histogram_df = pd.concat([histogram_df, histogram_df_append])
            
    plt.xlabel('Number of days (x) in the window')
    plt.ylabel('Number of people in x-day window')
    plt.ticklabel_format(scilimits = (0, 7))
    plt.title(f"Number of people in different # day windows for '{clause} x # days'")
    plt.hist(histogram_df['x'], 10)
    plt.show()

In [ ]:
# Finish setting up 30 day sample
df_30_days = duckdb.query(f"""
SELECT *
FROM df_30_days
WHERE (ABS("903115_datetime" - "1585723_datetime") <= 30) OR
      (ABS("903118_datetime" - "1585723_datetime") <= 30) OR
      (ABS("903126_datetime" - "1585723_datetime") <= 30) OR
      (ABS("903115_datetime" - "1585717_datetime") <= 30) OR
      (ABS("903118_datetime" - "1585717_datetime") <= 30) OR
      (ABS("903126_datetime" - "1585717_datetime") <= 30)
""").df()

print(f"Total number of people in 30 day sample (after filtering for the 30-day window): {df_30_days.shape[0]}")

In [ ]:
# Original dv sample

"""
Number of people in the dv sample: 28830

Proportion of females in the dv sample: 0.6895248005549774
Proportion of males in the dv sample: 0.30478668054110303

Proportion of White people in the dv sample: 0.7314255983350676
Proportion of Black or African American people in the dv sample: 0.08231009365244538
Proportion of people from more than one population in the dv sample: 0.056677072493929934
Proportion of Asian people in the dv sample: 0.02608394033992369
Proportion of people from another single population in the dv sample: 0.012833853624696497

Proportion of people who are not Hispanic or Latino in the dv sample: 0.8711758584807492
Proportion of people who are Hispanic or Latino in the dv sample: 0.10943461671869581

Mean age of dv sample: 48.22653485952133
Standard deviation of age for dv sample: 15.749750787255099
Proportion of missing age data in the dv sample: 0.0
"""

In [ ]:
# New dv sample (with doctor visits questions)
df_dv_new_doc = duckdb.query(f"""
SELECT *
FROM df_30_days
WHERE "What is the total number of visits to a mental health professional that you made in the last 12 months?" IS NOT NULL AND
      "What is the total number of general doctor visits you made in the last 12 months?" IS NOT NULL AND
      "How often do you feel deep inner peace or harmony?" IS NOT NULL AND
      "Computed diastolic blood pressure, mean of 2nd and 3rd measures" IS NOT NULL AND
      "Computed systolic blood pressure, mean of 2nd and 3rd measures" IS NOT NULL AND
      "Computed heart rate, mean of 2nd and 3rd measures" IS NOT NULL AND
      "What is your annual household income from all sources?" IS NOT NULL
""").df()

# New dv sample (with social item scores)
df_dv_new_soc = duckdb.query(f"""
SELECT *
FROM df_30_days
WHERE "How often do you feel deep inner peace or harmony?" IS NOT NULL AND
      "Computed diastolic blood pressure, mean of 2nd and 3rd measures" IS NOT NULL AND
      "Computed systolic blood pressure, mean of 2nd and 3rd measures" IS NOT NULL AND
      "Computed heart rate, mean of 2nd and 3rd measures" IS NOT NULL AND
      "What is your annual household income from all sources?" IS NOT NULL AND
      "Social support score" IS NOT NULL AND
      "Loneliness score" IS NOT NULL AND
      "PSS" IS NOT NULL
""").df()

# New dv sample (with doctor visits questions and social item scores)
df_dv_new_doc_soc = duckdb.query(f"""
SELECT *
FROM df_30_days
WHERE "What is the total number of visits to a mental health professional that you made in the last 12 months?" IS NOT NULL AND
      "What is the total number of general doctor visits you made in the last 12 months?" IS NOT NULL AND
      "How often do you feel deep inner peace or harmony?" IS NOT NULL AND
      "Computed diastolic blood pressure, mean of 2nd and 3rd measures" IS NOT NULL AND
      "Computed systolic blood pressure, mean of 2nd and 3rd measures" IS NOT NULL AND
      "Computed heart rate, mean of 2nd and 3rd measures" IS NOT NULL AND
      "What is your annual household income from all sources?" IS NOT NULL AND
      "Social support score" IS NOT NULL AND
      "Loneliness score" IS NOT NULL AND
      "PSS" IS NOT NULL
""").df()

In [ ]:
# Tuple that includes all the independent variables used in the analysis
ivs = ('In general, would you say your health is:',
       'In general, would you say your quality of life is:',
       'In general, how would you rate your physical health?',
       'In general, how would you rate your mental health, including your mood and your ability to think?',
       'In general, how would you rate your satisfaction with your social activities and relationships?',
       'In general, please rate how well you carry out your usual social roles.',
       'To what extent are you able to carry out your everyday physical activities such as walking, climbing stairs, carrying groceries, or moving a chair?',
       'In the past 7 days, how often have you been bothered by emotional problems such as feeling anxious, depressed or irritable?',
       'In the past 7 days, how would you rate your fatigue on average?',
       'In the past 7 days, how would you rate your pain on average?',
       'Global Physical Health Sum',
       'Global Mental Health Sum',
       'Global Physical Health T-Score',
       'Global Mental Health T-Score')

# Tuple that includes all the dependent variables used in the analysis
dvs = ('What is the total number of visits to a mental health professional that you made in the last 12 months?',
       'What is the total number of general doctor visits you made in the last 12 months?',
       'How often do you feel deep inner peace or harmony?',
       'Computed diastolic blood pressure, mean of 2nd and 3rd measures',
       'Computed systolic blood pressure, mean of 2nd and 3rd measures',
       'Computed heart rate, mean of 2nd and 3rd measures',
       'What is your annual household income from all sources?',
       'Social support score',
       'Loneliness score',
       'PSS')

In [ ]:
# Descriptives for each of the samples

samples = (df_big, df_30_days, df_dv_new_doc, df_dv_new_soc, df_dv_new_doc_soc)
samples_str = ('df_big', 'df_30_days', 'df_dv_new_doc', 'df_dv_new_soc', 'df_dv_new_doc_soc') # Necessary in order for the 'from' statements to work
sample_names = ('bigger sample', 
                'sample that only includes people who had a physical measurement within 30 days',
                'new dv sample (with doctor visits questions)',
                'new dv sample (with social item scores)',
                'new dv sample (with doctor visits questions and social item scores)')

for i in range(len(samples)):
    
    if i < 2: # Only interested in calculating proportions of missing data for the first two samples
        for j in ivs[0:10]:
            print(duckdb.query(f"""
            SELECT COUNT(*) / {samples[i].shape[0]} AS "Proportion of missing data for '{j}' in the {sample_names[i]}"
            FROM {samples_str[i]}
            WHERE "{j}" IS NULL
            """))
        for j in dvs:
            print(duckdb.query(f"""
            SELECT COUNT(*) / {samples[i].shape[0]} AS "Proportion of missing data for '{j}' in the {sample_names[i]}"
            FROM {samples_str[i]}
            WHERE "{j}" IS NULL
            """))

    print(f"Number of people in the {sample_names[i]}: {samples[i].shape[0]}")
    print('')
    print(f"Proportion of females in the {sample_names[i]}: {samples[i]['sex_at_birth'].value_counts()['Female'] / samples[i].shape[0]}")
    print(f"Proportion of males in the {sample_names[i]}: {samples[i]['sex_at_birth'].value_counts()['Male'] / samples[i].shape[0]}")
    print('')
    print(f"Proportion of White people in the {sample_names[i]}: {samples[i]['race'].value_counts()['White'] / samples[i].shape[0]}")
    print(f"Proportion of Black or African American people in the {sample_names[i]}: {samples[i]['race'].value_counts()['Black or African American'] / samples[i].shape[0]}")
    print(f"Proportion of people from more than one population in the {sample_names[i]}: {samples[i]['race'].value_counts()['More than one population'] / samples[i].shape[0]}")
    print(f"Proportion of Asian people in the {sample_names[i]}: {samples[i]['race'].value_counts()['Asian'] / samples[i].shape[0]}")
    print(f"Proportion of people from another single population in the {sample_names[i]}: {samples[i]['race'].value_counts()['Another single population'] / samples[i].shape[0]}")
    print('')
    print(f"Proportion of people who are not Hispanic or Latino in the {sample_names[i]}: {samples[i]['ethnicity'].value_counts()['Not Hispanic or Latino'] / samples[i].shape[0]}")
    print(f"Proportion of people who are Hispanic or Latino in the {sample_names[i]}: {samples[i]['ethnicity'].value_counts()['Hispanic or Latino'] / samples[i].shape[0]}")
    print('')

    """
    Comparing date of birth against the date of a specific survey under the assumption that each survey and physical 
    measurement took place either on the same day or very close in time to each other. Specifically chose the survey question 
    I did because this survey question had no missing values for each of the samples and because it also wasn't one of the 
    variables that were modified to create the physical-measurement-within-30-days sample.
    """
    # Age calculation code
    df_age = samples[i][['date_of_birth', '1585729_datetime']]
    df_age = df_age.dropna()
    
    df_age['birth_year'] = df_age['date_of_birth'].dt.year
    df_age['birth_month_day'] = str(df_age['date_of_birth'].dt.month) + "/" + str(df_age['date_of_birth'].dt.day)
    
    df_age['survey_year'] = df_age['1585729_datetime'].dt.year
    df_age['survey_month_day'] = str(df_age['1585729_datetime'].dt.month) + "/" + str(df_age['1585729_datetime'].dt.day)
    
    df_age['age'] = np.where(df_age['survey_month_day'] >= df_age['birth_month_day'], 
                             df_age['survey_year'] - df_age['birth_year'],
                             df_age['survey_year'] - df_age['birth_year'] - 1)

    print(f"Mean age of {sample_names[i]}: {df_age['age'].mean()}")
    print(f"Standard deviation of age for {sample_names[i]}: {df_age['age'].std()}")
    print(f"Proportion of missing age data in the {sample_names[i]}: {(samples[i].shape[0] - df_age.shape[0]) / samples[i].shape[0]}")
    
    print('')
    print('')
    print('')
    print('')

In [ ]:
# This cell contains dataframes that no longer exist

"""
Two sample proportion test for determining significance of difference between the proportion of white people in the original 
dv sample and the proportion of white people in the bigger sample
"""
z, p = proportions_ztest([df_dv['race'].value_counts()['White'], df_big['race'].value_counts()['White']], 
                         [df_dv.shape[0], df_big.shape[0]])
print(f"p-value: {p}")

In [ ]:
# This cell contains dataframes that no longer exist

"""
Two sample proportion test for determining significance of difference between the proportion of black people in the original 
dv sample and the proportion of black people in the bigger sample
"""
z, p = proportions_ztest([df_dv['race'].value_counts()['Black or African American'], df_big['race'].value_counts()['Black or African American']], 
                         [df_dv.shape[0], df_big.shape[0]])
print(f"p-value: {p}")

In [ ]:
# Final cleaning code
df = df_30_days.copy()
ivs_list = list(ivs)
dvs_list = list(dvs)
ivs_list.extend(dvs_list)
df = df[ivs_list]

In [ ]:
# Histograms for each variable

for i in df.columns:
    if i not in ["Computed diastolic blood pressure, mean of 2nd and 3rd measures",
                 "Computed systolic blood pressure, mean of 2nd and 3rd measures",
                 "Computed heart rate, mean of 2nd and 3rd measures"]:
        plt.xlabel('Score')
        plt.ylabel('Number of people with score')
        plt.title(f"Scores for '{i}'")
        plt.hist(df[i])
        plt.show()
    else:
        plt.xlabel('Measurement')
        plt.ylabel('Number of people with measurement')
        plt.title(f"Measurements for '{i}'")
        plt.hist(df[i], 50)
        plt.show()

In [ ]:
# For each variable, see how many people have values that aren't NA (without requiring social support variable to be in 30 day window)

d_counts = {df.columns[i]:{'# of people':df.count()[i]} for i in range(len(df.columns))}
df_counts = pd.DataFrame(d_counts)
display(HTML(df_counts.to_html()))

print(f"Total number of people in 30 day sample (after filtering for the 30-day window): {df_30_days.shape[0]}")

In [ ]:
# For each variable, see how many people have values that aren't NA (with requiring social support variable to be in 30 day window)

df_reduced = duckdb.query(f"""
SELECT *
FROM df_30_days
WHERE (ABS("40192442_datetime" - "1585723_datetime") <= 30) OR
      (ABS("40192442_datetime" - "1585717_datetime") <= 30)
""").df()

print(f"Total number of people in 30 day sample (after filtering for the 30-day window and after requiring social support variable to be in 30 day window): {df_reduced.shape[0]}")

d_counts = {df_reduced.columns[i]:{'# of people':df_reduced.count()[i]} for i in range(len(df_reduced.columns))}
df_reduced_counts = pd.DataFrame(d_counts)
df_reduced_counts = df_reduced_counts.drop(columns = ['903115_datetime', '903118_datetime', '903126_datetime', 
                                                      '1585717_datetime', '1585723_datetime', '1585729_datetime',
                                                      '40192442_datetime', 'date_of_birth', 'race', 'ethnicity', 
                                                      'sex_at_birth'])
display(HTML(df_reduced_counts.to_html()))

In [ ]:
# Create the r2 matrix (replace 'pearson' with 'spearman' to view and create spearman r2 matrix)
r2_matrix = round(df.corr(method = 'pearson') ** 2, 2)
display(HTML(r2_matrix.to_html()))

In [ ]:
# Table that summarizes average r2s for each IV

# Only take averages for physical dvs
lists = ([], [], [], [], [], [], [], [], [], [], [], [], [], [])
avgs = []

for i in range(len(ivs)):
    for j in dvs:
        lists[i].append(r2_matrix[ivs[i]][j])
    avgs.append(round(np.mean(np.array([lists[i]])), 2))

d_avgr2s = {'In general, would you say your health is:':pd.Series([avgs[0]], index = ['Avg r2']),
            'In general, would you say your quality of life is:':{'Avg r2':avgs[1]},
            'In general, how would you rate your physical health?':{'Avg r2':avgs[2]},
            'In general, how would you rate your mental health, including your mood and your ability to think?':{'Avg r2':avgs[3]},
            'In general, how would you rate your satisfaction with your social activities and relationships?':{'Avg r2':avgs[4]},
            'In general, please rate how well you carry out your usual social roles.':{'Avg r2':avgs[5]},
            'To what extent are you able to carry out your everyday physical activities such as walking, climbing stairs, carrying groceries, or moving a chair?':{'Avg r2':avgs[6]},
            'In the past 7 days, how often have you been bothered by emotional problems such as feeling anxious, depressed or irritable?':{'Avg r2':avgs[7]},
            'In the past 7 days, how would you rate your fatigue on average?':{'Avg r2':avgs[8]},
            'In the past 7 days, how would you rate your pain on average?':{'Avg r2':avgs[9]},
            'Global Physical Health Sum':{'Avg r2':avgs[10]},
            'Global Mental Health Sum':{'Avg r2':avgs[11]},
            'Global Physical Health T-Score':{'Avg r2':avgs[12]},
            'Global Mental Health T-Score':{'Avg r2':avgs[13]}}

df_avgr2s = pd.DataFrame(d_avgr2s)
df_avgr2s

In [ ]:
# Create the correlation p-value matrix (replace 'pearson' with 'spearman' to view and create correlation p-value matrix under the spearman method)

corr_matrix = df.corr(method = 'pearson')

l = len(df.columns)
pval_matrix = pd.DataFrame(np.zeros((l, l)), index = df.columns, columns = df.columns)
mask = pd.notnull(df)

for i in range(l):
    maski = mask[df.columns[i]]
    for j in range(i + 1):
        pval_matrix[df.columns[i]][df.columns[j]] = pval_matrix[df.columns[j]][df.columns[i]] = f"({corr_matrix[df.columns[i]][df.columns[j]]}, {(maski & mask[df.columns[j]]).sum()})"

for col in pval_matrix.columns:
    pval_matrix[col] = pval_matrix[col].apply(calculate_p_value)
    
display(HTML(pval_matrix.to_html()))

In [ ]:
# Table that summarizes average p-values for each IV

lists = ([], [], [], [], [], [], [], [], [], [], [], [], [], [])
avgs = []

for i in range(len(ivs)):
    for j in dvs:
        lists[i].append(pval_matrix[ivs[i]][j])
    avgs.append(np.mean(np.array([lists[i]])))

d_avgpvals = {'In general, would you say your health is:':pd.Series([avgs[0]], index = ['Avg p-value']),
            'In general, would you say your quality of life is:':{'Avg p-value':avgs[1]},
            'In general, how would you rate your physical health?':{'Avg p-value':avgs[2]},
            'In general, how would you rate your mental health, including your mood and your ability to think?':{'Avg p-value':avgs[3]},
            'In general, how would you rate your satisfaction with your social activities and relationships?':{'Avg p-value':avgs[4]},
            'In general, please rate how well you carry out your usual social roles.':{'Avg p-value':avgs[5]},
            'To what extent are you able to carry out your everyday physical activities such as walking, climbing stairs, carrying groceries, or moving a chair?':{'Avg p-value':avgs[6]},
            'In the past 7 days, how often have you been bothered by emotional problems such as feeling anxious, depressed or irritable?':{'Avg p-value':avgs[7]},
            'In the past 7 days, how would you rate your fatigue on average?':{'Avg p-value':avgs[8]},
            'In the past 7 days, how would you rate your pain on average?':{'Avg p-value':avgs[9]},
            'Global Physical Health Sum':{'Avg p-value':avgs[10]},
            'Global Mental Health Sum':{'Avg p-value':avgs[11]},
            'Global Physical Health T-Score':{'Avg p-value':avgs[12]},
            'Global Mental Health T-Score':{'Avg p-value':avgs[13]}}

df_avgpvals = pd.DataFrame(d_avgpvals)
df_avgpvals

In [ ]:
# List of Global Physical Health items
gph_items = ['In the past 7 days, how would you rate your pain on average?', 
             'To what extent are you able to carry out your everyday physical activities such as walking, climbing stairs, carrying groceries, or moving a chair?',
             'In general, how would you rate your physical health?',
             'In the past 7 days, how would you rate your fatigue on average?']

# List of Global Mental Health items
gmh_items = ['In general, would you say your quality of life is:',
             'In general, how would you rate your mental health, including your mood and your ability to think?',
             'In general, how would you rate your satisfaction with your social activities and relationships?',
             'In the past 7 days, how often have you been bothered by emotional problems such as feeling anxious, depressed or irritable?']

In [ ]:
# Global Physical Health items predicting DVs
for i in dvs:
    gph_items.append(i)
    sub_df = df[gph_items].dropna()
    gph_items.remove(i)
    x = sm.add_constant(sub_df[gph_items])
    y = sub_df[[i]]
    model = sm.OLS(y, x).fit()
    print(f"Model summary for Global Physical Health items predicting '{i}':")
    print(model.summary())
    print('')
    print('')
    print('')
    print('')

In [ ]:
# Global Mental Health items predicting DVs
for i in dvs:
    gmh_items.append(i)
    sub_df = df[gmh_items].dropna()
    gmh_items.remove(i)
    x = sm.add_constant(sub_df[gmh_items])
    y = sub_df[[i]]
    model = sm.OLS(y, x).fit()
    print(f"Model summary for Global Mental Health items predicting '{i}':")
    print(model.summary())
    print('')
    print('')
    print('')
    print('')

In [ ]:
# Raw sums predicting DVs
for i in dvs:
    sums.append(i)
    sub_df = df[sums].dropna()
    sums.remove(i)
    x = sm.add_constant(sub_df[sums])
    y = sub_df[[i]]
    model = sm.OLS(y, x).fit()
    print(f"Model summary for raw sums predicting '{i}':")
    print(model.summary())
    print('')
    print('')
    print('')
    print('')

In [ ]:
# T-Scores predicting DVs
for i in dvs:
    tscores.append(i)
    sub_df = df[tscores].dropna()
    tscores.remove(i)
    x = sm.add_constant(sub_df[tscores])
    y = sub_df[[i]]
    model = sm.OLS(y, x).fit()
    print(f"Model summary for T-Scores predicting '{i}':")
    print(model.summary())
    print('')
    print('')
    print('')
    print('')

In [ ]:
# Global Physical Health and Global Mental Health items predicting DVs

items = list(ivs[1:5])
items.extend(list(ivs[6:10]))

for i in dvs:

    items.append(i)
    sub_df = df[items].dropna()
    items.remove(i)
    x_lm = sm.add_constant(sub_df[items])
    x = sub_df[items]
    y = sub_df[[i]]
    k_folds = KFold(n_splits = 5)
    lin_reg = sm.OLS(y, x_lm).fit()
    lin_reg_cross_val = linear_model.LinearRegression().fit(x, y) # For comparing linear regression to other ML models (cross_val_score() does not take statsmodels objects)
    lasso = linear_model.Lasso().fit(x, y)
    elastic_net = linear_model.ElasticNet().fit(x, y)
    models = [lin_reg_cross_val, lasso, elastic_net]
    model_names = ['Linear Regression', 'Lasso', 'Elastic Net']
    rmse_best = -1

    for j in range(0, len(models)):

        # Use cross validation to compare multiple different types of ML models
        scores = cross_val_score(models[j], x, y, cv = k_folds, scoring = 'neg_mean_squared_error')
        rmse = np.sqrt(np.abs(np.mean(np.array([scores]))))

        if rmse_best == -1 or rmse < rmse_best:
            model_best_name = model_names[j]
            model_best = models[j]
            rmse_best = rmse

    print(f"Best model: {model_best_name}")

    if model_best_name == 'Linear Regression':
        model_best = lin_reg # If the best model is the linear model, use the linear model with OLS estimation
        print(f"Model summary for Global Physical Health and Global Mental Health items predicting '{i}':")
        print(model_best.summary())
    else:
        print(f"Equation of best model for Global Physical Health and Global Mental Health items predicting '{i}':")
        print(f"y = {model_best.coef_[0][0]}x1 + {model_best.coef_[0][1]}x2 + {model_best.coef_[0][2]}x3 + {model_best.coef_[0][3]}x4 + {model_best.coef_[0][4]}x5 + {model_best.coef_[0][5]}x6 + {model_best.coef_[0][6]}x7 + {model_best.coef_[0][7]}x8 + {model_best.coef_[0][8]}x9 + {model_best.coef_[0][9]}x10 + {model_best.intercept_[0]}")
        print('')
        print(f"RMSE of best model for Global Physical Health and Global Mental Health items predicting '{i}':")
        print(rmse_best)

    print('')
    print('')
    print('')
    print('')